#### https://github.com/HVL-ML/DAT255/blob/main/notebooks/15_gradio_and_streamlit.ipynb er brukt som utgangspunkt i denne notebooken.

# Gradio (and Streamlit) deployment

For deploying an ML-based app there are various approaches, but [Gradio](gradio.app) and [Streamlit](https://streamlit.io/) are both quick and convenient ways to do so. Typically we would implement this in a plain python (`.py`) file rather than a `.ipynb` notebook, but Gradio works here too, so let's try that first. Streamlit needs to be run directly in python, but the code is given below, so you can try out that too.

In this example we set up an image classifier, where the user can upload an image and get the top 5 class predictions in return.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.applications.mobilenet_v2 import (
    MobileNetV2,
    preprocess_input,
    decode_predictions,
)
from PIL import Image

## Set up a pretrained model

Let's download and set up a `MobileNetV2` model, trained on the 1000 classes in the ImageNet dataset. You can change this to anything you like. Remeber, however, to also use the appropriate preprocessing function.

In [ ]:
# model = MobileNetV2(weights="imagenet")

# The model loading below is from Deep Learning with Python, Third edition. Chapter: 8, Fitting the model
from matplotlib import pyplot as plt


model = keras.models.load_model(
    "../models/hierarchical_cnn_baseline_hybrid_crop.keras"
)

def classify_image(img: Image.Image):

    # Resize to the input image to what MobileNet expects
    img = img.resize((300, 300))
    arr = np.array(img)

    # Check the color channels, so we can take both grayscale, RGB, and RGBA images as input.
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=-1)
    elif arr.shape[-1] == 4:
        arr = arr[..., :3]

    # Add batch dim, and preprocess
    arr = np.expand_dims(arr, axis=0)
    # arr = preprocess_input(arr.astype("float32"))

    # Predict!
    preds = model.predict(arr, verbose=0)
    car_model_preds = preds["lvl2"][0]
    return [{"Tesla": preds["lvl1"][0][0]}, {'3 2017–2023': car_model_preds[0], '3 2024–nå': car_model_preds[1], 'S 2012–2015': car_model_preds[2], 'S 2016–nå': car_model_preds[3], 'X': car_model_preds[4], 'Y 2020–2024': car_model_preds[5], 'Y 2025-nå': car_model_preds[6]}]



## Set up the Gradio interface

Check the [documentation](https://www.gradio.app/docs) for the various things we can add here.

In [10]:
import gradio as gr

# Example images
examples = [
    "https://cdn.britannica.com/79/232779-050-6B0411D7/German-Shepherd-dog-Alsatian.jpg",
    "https://cdn.britannica.com/41/126641-050-E1CA0E61/cat-suns-hill-Parthenon-Athens-Greece-Acropolis.jpg",
]

with gr.Blocks(title="Visual vehicle recognition in varying lighting conditions") as demo:
    gr.Markdown("## Visual vehicle recognition in varying lighting conditions")
    gr.Markdown(
        "Upload an image or click an example below. "
    )

    with gr.Row():
        image_input = gr.Image(type="pil", label="Opplastet bilde")
        label_output = [{},{}]
        label_output[0] = gr.Label(num_top_classes=1, label="Bilmerke")
        label_output[1] = gr.Label(num_top_classes=3, label="Bilmodell")

    classify_btn = gr.Button("Classify", variant="primary")
    classify_btn.click(fn=classify_image, inputs=image_input, outputs=label_output)

    gr.Examples(
        examples=examples,
        inputs=image_input,
        outputs=label_output,
        fn=classify_image,
        cache_examples=True
    )

Now we can run it:

In [11]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7863
Using cache from 'c:\Users\johan\Documents\School\Dat191\visual-vehicle-recognition-varying-lighting-conditions\gradio\.gradio\cached_examples\41' directory. If method or examples have changed since last caching, delete this folder to clear cache.

* To create a public link, set `share=True` in `launch()`.


[{'Tesla': np.float32(1.0)}, {'3 2017–2023': np.float32(0.0), '3 2024–nå': np.float32(0.0), 'S 2012–2015': np.float32(0.0), 'S 2016–nå': np.float32(0.0), 'X': np.float32(0.0), 'Y 2020–2024': np.float32(1.0), 'Y 2025-nå': np.float32(0.0)}]
[{'Tesla': np.float32(8.662804e-38)}, {'3 2017–2023': np.float32(0.0), '3 2024–nå': np.float32(0.0), 'S 2012–2015': np.float32(0.0), 'S 2016–nå': np.float32(0.0), 'X': np.float32(0.0), 'Y 2020–2024': np.float32(1.0), 'Y 2025-nå': np.float32(0.0)}]
[{'Tesla': np.float32(0.0)}, {'3 2017–2023': np.float32(0.0), '3 2024–nå': np.float32(0.0), 'S 2012–2015': np.float32(0.0), 'S 2016–nå': np.float32(0.0), 'X': np.float32(0.0), 'Y 2020–2024': np.float32(1.0), 'Y 2025-nå': np.float32(0.0)}]
[{'Tesla': np.float32(0.0)}, {'3 2017–2023': np.float32(7.8186804e-36), '3 2024–nå': np.float32(0.0), 'S 2012–2015': np.float32(3.50495e-17), 'S 2016–nå': np.float32(0.0), 'X': np.float32(0.0), 'Y 2020–2024': np.float32(1.0), 'Y 2025-nå': np.float32(0.0)}]
[{'Tesla': np.flo